# Ejemplos de Plots: M. Luz Clara (2026)

Para utilizar esta notebook es necesario haber corrido la primera notebook (Tutorial I). En esta vamos a encontrar como hacer distintos graficos/analisis a partir de los datasets generados en el:

Vamos a necestar los archivos (o fijarnos que nombre le pusimos a cada uno de esos archivos)

- pcaso_anual_stats.nc
- pcaso_seasonal_stats.nc
- pcaso_monthly_stats.nc
- pcaso_waves.nc

In [ ]:
import sys
import os

import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats

try:
    """
    Si la libreria esta instalada
    """
    import mhwpy.datasets_utils as dsu # control de dataset facilitando el uso, no obligatorio
    import mhwpy.mhw_core as mhw # core del script 
    import mhwpy.time_utils as timeu # herramientas y utilidades del manejo del tiempo
    
except ImportError:
    """
    Si la libreria no se instalado o no desea ser instalada. Importante,
    esto es posible si y solo si la notebook esta ubicada en ./mwh/notebooks/
    """
    # Ruta ubicacion scripts mhw
    ruta_mhw = os.path.abspath(os.path.join('..', 'mhw'))

    # Agregamos la ruta al sistema
    if ruta_mhw not in sys.path:
        sys.path.append(ruta_mhw)
    
    import datasets_utils as dsu
    import time_utils as timeu
    from stats_and_trends import trend_matrix

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.axes as maxes
# Cartopy 
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.crs import PlateCarree
from cartopy.feature import LAND, COASTLINE, RIVERS
# Beatiful ocean
from cmocean.cm import haline, thermal

import warnings
# Ignora warnings especificamente de numpy y la existencia de arrays con nans
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

%load_ext autoreload
%autoreload 2

In [ ]:
nf_anual = "pcaso_anual_stats.nc"
nf_seasonal = "pcaso_seasonal_stats.nc"
nf_monthly = "pcaso_monthly_stats.nc"
nf_waves = "pcaso_waves.nc"
nf_climdata = "pcaso_clim.nc"

### Ejemplo I: Grafico de eventos totales, duracion media, maxima intensidad e intensidad media

In [ ]:
# cargamos dataset anual
ds_anual = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_anual))

In [ ]:
# sumamos sobre "axis tiemepo", obteniendo una grilla de eventos 
eventos_totales = ds_anual.total_events.sum(axis=0, skipna=True)
# calculamos el maximo de anomalia observada por pixel
anomalias_maximas = ds_anual.anom_max.max(axis=0, skipna=True)
# calculamos el premedio de las anomalias maximas observadas (anio a anio)
anomalias_maximas_media = ds_anual.anom_max.mean(axis=0, skipna=True)
# calculamos la duracion media de los eventos (anio a anio)
duration_mean = ds_anual.duration.mean(axis=0, skipna=True)
# *importante en M. Luz Clara et al la estimacion de algunas de estas metricas fue levemente diferente

In [ ]:
lat = ds_anual.lat.values
lon = ds_anual.lon.values - 360

In [ ]:
pad = 0.07
shrink = 0.77
color = "k"

fig = plt.figure(figsize=(7, 8))

ax1 = fig.add_axes([0., 0.5, 0.5, 0.42], projection=PlateCarree())
ax4 = fig.add_axes([0.5, 0.5, 0.5, 0.42], projection=PlateCarree())
ax2 = fig.add_axes([0., 0, 0.5, 0.42], projection=PlateCarree())
ax3 = fig.add_axes([0.5, 0, 0.5, 0.42], projection=PlateCarree())

ax1.set_facecolor('white')
ax1.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax1.add_feature(COASTLINE, linewidth=0.5)
ax1.set_title("Total MHW ", size=15)

ax2.set_facecolor('white')
ax2.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax2.add_feature(COASTLINE, linewidth=0.5)
ax2.set_title("Max MHW intensity", size=15)

ax3.set_facecolor('white')
ax3.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax3.add_feature(COASTLINE, linewidth=0.5)
ax3.set_title("Mean max MHW intensity", size=15)

ax4.set_facecolor('white')
ax4.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax4.add_feature(COASTLINE, linewidth=0.5)
ax4.set_title("Mean MHW duration", size=15)
contorno = False

im = ax1.contourf(lon, lat, eventos_totales , levels=np.arange(60, 120, 0.01), cmap="OrRd", zorder=0)
clb = plt.colorbar(im, ticks=range(60, 120, 5), shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
clb.set_label("Events", size=12)
clb.ax.tick_params(rotation=45)


im = ax2.contourf(lon, lat, anomalias_maximas, levels=np.arange(0, 10.001, 0.01), cmap="OrRd", zorder=0)
clb = plt.colorbar(im, ticks=np.arange(0, 11, 1),  shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
clb.set_label("°C", size=12)


im = ax3.contourf(lon, lat, anomalias_maximas_media , levels=np.arange(0, 10.001, 0.01),  cmap="OrRd", zorder=0)
clb = plt.colorbar(im, ticks=np.arange(0, 11, 1),  shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
clb.set_label("°C", size=12)

im = ax4.contourf(lon, lat, duration_mean, levels=np.arange(5, 23.1, 0.1),  cmap="OrRd", zorder=0)
clb = plt.colorbar(im, ticks=range(5, 25, 2), shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
ax4.clabel(im, fontsize=7)
clb.set_label("Days", size=12)


ax1.set_ylim([-54.7, -33.4])
ax1.set_xlim([290 - 360, 311 - 360])

ax2.set_ylim([-54.7, -33.4])
ax2.set_xlim([290 - 360, 311 - 360])

ax3.set_ylim([-54.7, -33.4])
ax3.set_xlim([290 - 360, 311 - 360])

ax4.set_ylim([-54.7, -33.4])
ax4.set_xlim([290 - 360, 311 - 360])


yticks = np.arange(-54, -33, 3)
xticks = np.arange(-69, -50, 3)

gl = ax1.gridlines(crs=PlateCarree(), draw_labels=True, linestyle='--', linewidth=0, zorder=1)
gl.yformatter = LATITUDE_FORMATTER
gl.xformatter = LONGITUDE_FORMATTER
gl.right_labels = True
gl.top_labels = False
gl.left_labels = False

gl = ax2.gridlines(crs=PlateCarree(), draw_labels=True, linestyle='--', linewidth=0, zorder=1)
gl.yformatter = LATITUDE_FORMATTER
gl.xformatter = LONGITUDE_FORMATTER
gl.right_labels = True
gl.top_labels = False
gl.left_labels = False

gl = ax3.gridlines(crs=PlateCarree(), draw_labels=True, linestyle='--', linewidth=0, zorder=1)
gl.yformatter = LATITUDE_FORMATTER
gl.xformatter = LONGITUDE_FORMATTER
gl.right_labels = True
gl.top_labels = False
gl.left_labels = False

gl = ax4.gridlines(crs=PlateCarree(), draw_labels=True, linestyle='--', linewidth=0, zorder=0)
gl.yformatter = LATITUDE_FORMATTER
gl.xformatter = LONGITUDE_FORMATTER
gl.right_labels = True
gl.top_labels = False
gl.left_labels = False

ax1.set_yticks(yticks, crs=PlateCarree())
ax1.set_yticklabels(labels=[""] * len(yticks))
ax1.yaxis.set_ticks_position('right')
ax1.set_xticks(xticks, crs=PlateCarree())
ax1.set_xticklabels(labels=[""] * len(xticks))

ax2.set_yticks(yticks, crs=PlateCarree())
ax2.set_yticklabels(labels=[""] * len(yticks))
ax2.yaxis.set_ticks_position('right')
ax2.set_xticks(xticks, crs=PlateCarree())
ax2.set_xticklabels(labels=[""] * len(xticks))

ax3.set_yticks(yticks, crs=PlateCarree())
ax3.set_yticklabels(labels=[""] * len(yticks))
ax3.yaxis.set_ticks_position('right')
ax3.set_xticks(xticks, crs=PlateCarree())
ax3.set_xticklabels(labels=[""] * len(xticks))

ax4.set_yticks(yticks, crs=PlateCarree())
ax4.set_yticklabels(labels=[""] * len(yticks))
ax4.yaxis.set_ticks_position('right')
ax4.set_xticks(xticks, crs=PlateCarree())
ax4.set_xticklabels(labels=[""] * len(xticks))

ax1.text(-69, -36, "(a)", fontdict={"size": 20})
ax2.text(-69, -36, "(b)", fontdict={"size": 20})
ax3.text(-69, -36, "(c)", fontdict={"size": 20})
ax4.text(-69, -36, "(d)", fontdict={"size": 20})

plt.show()

### Ejemplo II: Tendencia de eventos anuales y estacionales

In [ ]:
ds_seasonal = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_seasonal))

In [ ]:
pv_treshold = 1

In [ ]:
trend, pvalue, intercept = trend_matrix(ds_anual.total_events, time_dim="year")
anual_trend = np.where(pvalue < pv_treshold, trend, np.nan)

In [ ]:
trend, pvalue, intercept = trend_matrix(ds_seasonal.total_events[:, 0, :, :], time_dim="year")
auntumn_trend = np.where(pvalue < pv_treshold, trend, np.nan)

trend, pvalue, intercept = trend_matrix(ds_seasonal.total_events[:, 1, :, :], time_dim="year")
spring_trend = np.where(pvalue < pv_treshold, trend, np.nan)

trend, pvalue, intercept = trend_matrix(ds_seasonal.total_events[:, 2, :, :], time_dim="year")
summer_trend = np.where(pvalue < pv_treshold, trend, np.nan)

trend, pvalue, intercept = trend_matrix(ds_seasonal.total_events[:, 3, :, :], time_dim="year")
winter_trend = np.where(pvalue < pv_treshold, trend, np.nan)

In [ ]:
shrink = 1
pad = 0.04
color = "k"

levels = np.arange(-.4, .41, .01)
ticks =  np.arange(-.4, .6, .1)

levels2 = np.arange(-.11, .11, .01)
ticks2 = np.arange(-.15, .16, .05)


fig = plt.figure(figsize=(12, 8))
ax0 = fig.add_axes([0, 0., .15, 1], projection=PlateCarree()) 
ax2 = fig.add_axes([0.2, 0., .15, 1], projection=PlateCarree()) 
ax3 = fig.add_axes([0.4, 0., .15, 1], projection=PlateCarree()) 
ax1 = fig.add_axes([0.6, 0., .15, 1], projection=PlateCarree()) 
ax4 = fig.add_axes([0.8, 0., .15, 1], projection=PlateCarree()) 

ax0.set_facecolor('white')
ax0.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax0.add_feature(COASTLINE, linewidth=0.5)
ax0.set_title("Total MHW ", size=15)

ax1.set_facecolor('white')
ax1.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax1.add_feature(COASTLINE, linewidth=0.5)
ax1.set_title("Total MHW ", size=15)

ax2.set_facecolor('white')
ax2.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax2.add_feature(COASTLINE, linewidth=0.5)
ax2.set_title("Total MHW ", size=15)

ax3.set_facecolor('white')
ax3.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax3.add_feature(COASTLINE, linewidth=0.5)
ax3.set_title("Total MHW ", size=15)

ax4.set_facecolor('white')
ax4.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax4.add_feature(COASTLINE, linewidth=0.5)
ax4.set_title("Total MHW ", size=15)

im = ax0.contourf(lon, lat, anual_trend, levels=levels, cmap="RdBu_r", zorder=0)
clb = plt.colorbar(im, ticks=ticks, shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
#clb.set_label("Events/Decade", size=18)
clb.ax.tick_params(rotation=90)
clb.ax.set_xlabel(r"$\frac{events}{year}$", size=15)


im = ax1.contourf(lon, lat, auntumn_trend, levels=levels2, cmap="RdBu_r", zorder=0)
clb = plt.colorbar(im, ticks=ticks2, shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
clb.ax.tick_params(rotation=90)
clb.ax.set_xlabel(r"$\frac{events}{year}$", size=15)


im = ax2.contourf(lon, lat, spring_trend, levels=levels2, cmap="RdBu_r", zorder=0)
clb = plt.colorbar(im, ticks=ticks2, shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
#clb.set_label("Events/Decade", size=18)
clb.ax.tick_params(rotation=90)
clb.ax.set_xlabel(r"$\frac{events}{year}$", size=15)


im = ax3.contourf(lon, lat, summer_trend, levels=levels2, cmap="RdBu_r", zorder=0)
clb = plt.colorbar(im, ticks=ticks2, shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
#clb.set_label("Events/Decade", size=18)
clb.ax.tick_params(rotation=90)
clb.ax.set_xlabel(r"$\frac{events}{year}$", size=15)


im = ax4.contourf(lon, lat, winter_trend, levels=levels2, cmap="RdBu_r", zorder=0)
clb = plt.colorbar(im, ticks=ticks2, shrink=shrink, pad=pad , extend='both', drawedges=False, orientation="horizontal")
#clb.set_label("Events/Decade", size=18)
clb.ax.tick_params(rotation=90)
clb.ax.set_xlabel(r"$\frac{events}{year}$", size=15)

ax0.set_title("Year")
ax1.set_title("Autumn")
ax2.set_title("Spring")
ax3.set_title("Summer")
ax4.set_title("Winter")

fig.suptitle(f"Tendencia de eventos (p_valor = {pv_treshold})", x=0.4, y=0.5, c="red", fontsize=20)
plt.show()

Aclaracion: en M. Luz Clara et al (2026) la tendencia se calcula a partir del 2018, en este ejemplo tomamos todo el periodo.

### Ejemplo III: Heatmap % de area cubierta por olas de calor (en una dada subregion)

In [ ]:
ds_monthly = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_monthly))

In [ ]:
region_bands = [299, 302, -44.0, -40.0] 

# Limito los datos por LAT/LON 
cond_lat = (ds_monthly.lat > region_bands[2]) & (ds_monthly.lat < region_bands[3])
cond_lon = (ds_monthly.lon > region_bands[0]) & (ds_monthly.lon < region_bands[1])
ds_area = ds_monthly.where(cond_lon & cond_lat, drop=True)

In [ ]:
total_events = ds_area.total_events
pixel_we = np.where(total_events.isnull(), 0, 1)

In [ ]:
total_pixels = pixel_we.shape[2] *  pixel_we.shape[3]
ratio_area = 100 * pixel_we.sum(axis=(2, 3))/total_pixels 

In [ ]:
ds_area.year.values.max(), ds_area.year.values.min()

In [ ]:
fig = plt.figure(figsize=(14, 3))
ax = fig.add_axes([0, 0.0, 1, 1], ) 
sns.heatmap(ratio_area.T, 
            linewidths=.5, 
            cmap="Reds", 
            vmin=0, vmax=100, 
            ax=ax, 
            annot=True,
            cbar_kws={"pad": 0.005, "label":"%"})

ax.set_yticks(np.arange(0.5, 12.5, 1), labels=range(1, 13, 1))
ax.set_xticks(np.arange(0.5, 43.5, 1), labels=range(1982, 2025, 1), rotation=90)
ax.set_ylabel("MES")
ax.set_title("Area (%) con eventos de olar de calor")
plt.show()

### Ejemplo IV: MovieMaker - Visualizando las olas de calor

Agregamos como ejemplo como realizar gif para el seguimiento de olas de calor. El dataset llamado "waves" es una mascaras de 0 y 1. El dataset clim tiene las variables termicas asociadas, en este caso, nos va interesar ambas. 

In [ ]:
# si las liberias no estan instaladas, instalarlas usando pip
from mpl_toolkits.axes_grid1 import make_axes_locatable
from moviepy.editor import VideoClip
from moviepy.video.io.bindings import mplfig_to_npimage


In [ ]:
# cargamos datasets necesarios
ds_waves = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_waves))
ds_climdata = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_climdata))

ds_anomwaves = (ds_waves.waves * ds_climdata.anomalies).copy()

ds_waves.close()
ds_climdata.close()

In [ ]:
# tomamos solo un mes 
date_init = "1998-10-01T00:00:00"
date_end = "1998-10-31T23:59:59"

cond_time_inf = ds_anomwaves.time >= np.datetime64(date_init) 
cond_time_sup = ds_anomwaves.time <= np.datetime64(date_end)

ds_period = ds_anomwaves.where((cond_time_inf & cond_time_sup), drop=True)

In [ ]:
location="right"
size="5%"
pad="2%"

duration = 31
fps = 1


fig = plt.figure(figsize=(6, 4))

ax = fig.add_axes([0., 0, .9, .9], projection=PlateCarree())
ax.set_facecolor('white')
ax.add_feature(LAND, facecolor="gainsboro", zorder=2)
ax.add_feature(COASTLINE, linewidth=0.5)
ax.set_title("Total MHW ", size=15)


divider = make_axes_locatable(ax)
cax = divider.append_axes(location, size=size, pad=size, axes_class=maxes.Axes)
contourf = ax.contourf(lon, lat, ds_period.values[0, :, :], 
                       cmap="gist_ncar",  
                       levels=np.arange(0, 5.1, .1), 
                       zorder=0)
cbar = fig.colorbar(contourf,  cax=cax, label="Anomalia (°C)")
#plot_region(ax, region=region, color="k")


def make_frame(t):
    index = int(fps * t)
    for coll in ax.collections:
        coll.remove()
    ax.contourf(lon, lat, ds_period.values[index, :, :], 
                cmap="gist_ncar", 
                levels=np.arange(0, 5.1, 0.1),
                zorder=0)
    ax.set_title(f"Anomalias\n 1998 - Octubre - {index + 1}")
    
    g = ax.gridlines(crs=PlateCarree(), draw_labels=True,  zorder=0)
    g.ylabels_right = False
    g.xlabels_top = False
    g.yformatter = LATITUDE_FORMATTER
    g.xformatter = LONGITUDE_FORMATTER
    g.xlabel_style={'size':10}
    g.ylabel_style={'size':10}
    return mplfig_to_npimage(fig)

In [ ]:
animation = VideoClip(make_frame, duration=duration)
nf = "./datasets/PCASO_anomalias_octubre_1998.mp4"
animation.write_videofile(nf, fps=1, logger=None)